# Blood Cell Classification — starter notebook

Classify microscopy images of **blood cells** into one of **8 cell types**, from
28×28 RGB images.

This notebook runs end-to-end: **download the data → train a simple model → submit**.
Metric: **F1-macro**.

> Open in Colab, then run the cells top to bottom. The only thing you must edit is
> your API token in the next cell (find it on your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q mlarena-sdk scikit-learn pandas numpy  # installs the `mlarena` package

import mlarena
import numpy as np
import pandas as pd

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
COMPETITION_ID = 173

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`download_dataset` pulls the competition files into `data/`. Each image is a
`28 × 28 × 3` (RGB) array; the training labels are in `y_train.csv` (`image_id,label`).

In [ ]:
client.download_dataset(COMPETITION_ID, "data/")

train = np.load("data/train_images.npz")   # train["image_id"], train["images"] (N,28,28,3) uint8
test  = np.load("data/test_images.npz")    # test["image_id"],  test["images"]
y_train = pd.read_csv("data/y_train.csv")  # image_id,label

# line up each image with its label (match on image_id, don't assume row order)
label_of = y_train.set_index("image_id")["label"]
y = label_of.loc[train["image_id"]].values

print("train images:", train["images"].shape, "  test images:", test["images"].shape)
print(pd.Series(y).value_counts())

## 2. A baseline model

Flatten each image into a vector of pixels and fit a random forest. This is just a
starting point — a small CNN (e.g. PyTorch) trained on the raw images will score
much higher. We hold out 20% to estimate the F1-macro before submitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X = train["images"].reshape(len(train["images"]), -1) / 255.0   # (N, 28*28*3)
X_test = test["images"].reshape(len(test["images"]), -1) / 255.0

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
clf.fit(X_tr, y_tr)
print("validation F1-macro: %.4f" % f1_score(y_val, clf.predict(X_val), average="macro"))

clf.fit(X, y)   # refit on all the training data before predicting the test set

## 3. Predict and write `submission.csv`

In [ ]:
pred = clf.predict(X_test)
submission = pd.DataFrame({"image_id": test["image_id"], "label": pred})
submission.to_csv("submission.csv", index=False)
submission.head()

## 4. Submit

Run the cell below to submit through the SDK, or upload `submission.csv` on the
competition page.

In [ ]:
result = client.submit(competition_id=COMPETITION_ID, files=["submission.csv"])
print(result)

## 5. Where to go from here

The pixel-flattening baseline throws away the *spatial* structure of the image — that's
where most of the score is hiding. A good loop is:
**change one thing → watch the validation F1 → submit if it helps.** Some directions to try:

- **Look at the images first.** Plot a few per class, check class balance, spot anything odd.
- **Train a small CNN (PyTorch).** A couple of conv + pooling layers already beats the random
  forest by a wide margin.
- **Train carefully.** Keep a validation set, use **early stopping**, and tune the
  **learning rate** (the single most important knob), plus batch size and epochs.
- **Augment the data.** Flips, rotations, and small color jitter help the model generalize.
- **Use a pretrained model.** Load a network pretrained on ImageNet (ResNet, EfficientNet…)
  and **fine-tune** it on these cells — usually the biggest jump for the least effort.

Watch train vs. validation: if validation stalls while train keeps improving, you're
overfitting — stop early or regularize.